# 02. Norman benchmark: step by step

This notebook is the original, cell-by-cell version of what `cellcheck run` does.
It loads the Norman et al. 2019 dataset, builds simple baselines, tests Geneformer's gene embeddings with a
linear model, adds controls, and repeats everything over 30 random splits.

Nothing is trained or fine-tuned except a small ridge regression. Outputs are written to `/kaggle/working/cellcheck/`.

## 1. Load the data

In [ ]:
!pip -q install pertpy 2>&1 | tail -n 3

In [ ]:
import os, logging
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
logging.getLogger("httpx").setLevel(logging.WARNING)   # hides the HTTP spam

import pertpy as pt
adata = pt.data.norman_2019()

print(adata)
print("\nobs columns:", adata.obs.columns.tolist())
print("\nvar columns:", adata.var.columns.tolist())
print("\nfirst gene names:", adata.var_names[:5].tolist())
print("max value in X:", adata.X.max())
print("\nexample cell rows:")
print(adata.obs.head(3).T)

## 2. Count controls, singles and doubles; find Geneformer's dictionaries

In [ ]:
import numpy as np
from huggingface_hub import list_repo_files

names = adata.obs["perturbation_name"].astype(str)
is_ctrl = names.eq("control")
is_double = names.str.contains(r"\+")
is_single = ~is_ctrl & ~is_double

print("cells:", len(names))
print("control cells:", int(is_ctrl.sum()))
print("single-perturbation cells:", int(is_single.sum()), "| unique:", names[is_single].nunique())
print("double-perturbation cells:", int(is_double.sum()), "| unique:", names[is_double].nunique())
print("\ncells per perturbation:")
print(names[~is_ctrl].value_counts().describe().round(1))

# are the 'counts' real integer counts?
sample = adata.layers["counts"][:1000]
sample = sample.toarray() if hasattr(sample, "toarray") else sample
print("\ncounts are integers:", bool(np.allclose(sample, sample.round())), "| max:", sample.max())

# where are Geneformer's gene dictionaries?
files = list_repo_files("ctheodoris/Geneformer")
hits = [f for f in files if any(k in f.lower() for k in ["dict", "token", "median", "ensembl"])]
print("\nGeneformer dictionary files:")
for f in hits[:40]:
    print(" ", f)

## 3. Which genes can Geneformer represent?

In [ ]:
import pickle
from huggingface_hub import hf_hub_download

def load(fname):
    path = hf_hub_download("ctheodoris/Geneformer", f"geneformer/{fname}")
    with open(path, "rb") as f:
        return pickle.load(f)   # official files from the model's own repo

name_to_id = load("gene_name_id_dict_gc104M.pkl")   # gene symbol -> Ensembl ID
token_dict = load("token_dictionary_gc104M.pkl")    # Ensembl ID -> token number
median_dict = load("gene_median_dictionary_gc104M.pkl")

for label, d in [("name_to_id", name_to_id), ("token_dict", token_dict), ("median_dict", median_dict)]:
    print(label, "| size:", len(d), "| sample:", list(d.items())[:3])

# 1) how many of Norman's genes does Geneformer know?
genes = list(adata.var_names)
mappable = [g for g in genes if g in name_to_id and name_to_id[g] in token_dict]
print(f"\nNorman genes Geneformer can tokenize: {len(mappable)} / {len(genes)}")

# 2) how many PERTURBED genes does it know?
pert_genes = set()
for n in names[~is_ctrl].unique():
    pert_genes.update(n.split("+"))
known = {g for g in pert_genes if g in name_to_id and name_to_id[g] in token_dict}
in_data = {g for g in pert_genes if g in set(genes)}
print(f"perturbed genes: {len(pert_genes)} | known to Geneformer: {len(known)} | measured in the data: {len(in_data)}")
print("missing from Geneformer:", sorted(pert_genes - known))

## 4. Baselines: no change and additive

Every method is scored on the same genes (the ones Geneformer can represent).

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, json

# --- gene set shared by ALL methods (fairness rule) ---
genes_ok = [g for g in adata.var_names if g in name_to_id and name_to_id[g] in token_dict]
col = {g: i for i, g in enumerate(adata.var_names)}
gidx = [col[g] for g in genes_ok]
print("var names unique:", adata.var_names.is_unique, "| shared genes:", len(gidx))

# --- one average profile per perturbation ("pseudobulk"), from the log-normalized matrix X ---
codes, uniq = pd.factorize(names)
G = sp.csr_matrix((np.ones(len(codes)), (codes, np.arange(len(codes)))),
                  shape=(len(uniq), len(codes)))
sums = G @ adata.X
sums = sums.toarray() if sp.issparse(sums) else np.asarray(sums)
counts = np.asarray(G.sum(1)).ravel()
pb = pd.DataFrame(sums[:, gidx] / counts[:, None], index=uniq, columns=genes_ok)
ctrl = pb.loc["control"].values

# --- which perturbations are usable? ---
usable = lambda p: all(g in known for g in p.split("+"))
singles = [p for p in pb.index if p != "control" and "+" not in p and usable(p)]
doubles = [p for p in pb.index if "+" in p and usable(p)
           and all(g in singles for g in p.split("+"))]   # both parts need their own single result
print("usable singles:", len(singles), "| usable doubles:", len(doubles))

# --- fixed test half of the doubles (saved so every future method uses the same one) ---
rng = np.random.default_rng(0)
test_doubles = sorted(rng.choice(doubles, size=len(doubles) // 2, replace=False).tolist())

# --- score each double ---
def corr(a, b):
    return np.nan if a.std() == 0 or b.std() == 0 else float(np.corrcoef(a, b)[0, 1])

rows = []
for d in doubles:
    a, b = d.split("+")
    truth = pb.loc[d].values
    preds = {
        "no_change": ctrl,
        "additive": ctrl + (pb.loc[a].values - ctrl) + (pb.loc[b].values - ctrl),
    }
    for m, p in preds.items():
        rows.append({"double": d, "method": m,
                     "rmse": float(np.sqrt(np.mean((p - truth) ** 2))),
                     "delta_corr": corr(p - ctrl, truth - ctrl),
                     "in_test_half": d in test_doubles})
res = pd.DataFrame(rows)

for label, sub in [("ALL usable doubles", res), ("TEST half only", res[res.in_test_half])]:
    print("\n" + label, "| n =", sub["double"].nunique())
    print(sub.groupby("method")[["rmse", "delta_corr"]].mean().round(4))

os.makedirs("/kaggle/working/cellcheck", exist_ok=True)
res.to_csv("/kaggle/working/cellcheck/baselines_norman_per_double.csv", index=False)
json.dump({"test_doubles": test_doubles, "shared_genes": len(genes_ok)},
          open("/kaggle/working/cellcheck/norman_split.json", "w"), indent=1)
print("\nsaved to /kaggle/working/cellcheck/")

## 5. Noise floor and a focused metric

The noise floor estimates the error that even a perfect predictor would show, using two random halves of the cells of each double.
The top-50 view picks the genes that change most in the real data, so it is a diagnostic and not a fair headline number.

In [ ]:
import numpy as np, scipy.sparse as sp

rng = np.random.default_rng(1)
Xcsr = adata.X.tocsr() if sp.issparse(adata.X) else sp.csr_matrix(adata.X)
nm = names.values

# 1) noise floor: split each double's cells in two halves and compare them.
#    A half has half the cells, so the error of a perfect predictor against the
#    full average is roughly (half-vs-half RMSE) / 2. This is an approximation.
floor = {}
for d in doubles:
    idx = np.where(nm == d)[0]
    rng.shuffle(idx)
    h = len(idx) // 2
    m1 = np.asarray(Xcsr[idx[:h]][:, gidx].mean(0)).ravel()
    m2 = np.asarray(Xcsr[idx[h:]][:, gidx].mean(0)).ravel()
    floor[d] = float(np.sqrt(np.mean((m1 - m2) ** 2)) / 2)

# 2) the same methods, scored on the 50 genes that change most in each double
#    (picked from the real data, a common way to focus on genes that respond)
rows = []
for d in doubles:
    a, b = d.split("+")
    truth = pb.loc[d].values
    top = np.argsort(-np.abs(truth - ctrl))[:50]
    preds = {"no_change": ctrl,
             "additive": ctrl + (pb.loc[a].values - ctrl) + (pb.loc[b].values - ctrl)}
    for m, p in preds.items():
        rows.append({"double": d, "method": m,
                     "rmse_top50": float(np.sqrt(np.mean((p[top] - truth[top]) ** 2))),
                     "delta_corr_top50": corr(p[top] - ctrl[top], truth[top] - ctrl[top]),
                     "in_test_half": d in test_doubles})
r2 = pd.DataFrame(rows)

print("Estimated noise floor (RMSE, all genes): mean",
      round(np.mean(list(floor.values())), 4), "| test half",
      round(np.mean([floor[d] for d in test_doubles]), 4))
print("\nTop-50 responding genes, TEST half:")
print(r2[r2.in_test_half].groupby("method")[["rmse_top50", "delta_corr_top50"]].mean().round(4))

pd.DataFrame({"double": list(floor), "noise_floor_rmse": list(floor.values())}) \
  .to_csv("/kaggle/working/cellcheck/noise_floor_norman.csv", index=False)
r2.to_csv("/kaggle/working/cellcheck/baselines_norman_top50.csv", index=False)

## 6. First test of Geneformer's gene embeddings

Each perturbation is described by the sum of the Geneformer vectors of its genes. A ridge regression predicts the change in expression.
Train on all singles and half of the doubles; test on the other half. Random and one-hot vectors act as controls.

In [ ]:
import numpy as np, pandas as pd, torch
from transformers import AutoModel
from sklearn.linear_model import RidgeCV

m = AutoModel.from_pretrained("ctheodoris/Geneformer", subfolder="Geneformer-V2-104M")
E = m.embeddings.word_embeddings.weight.detach().float().numpy()
del m
print("embedding table:", E.shape)

pgenes = sorted({g for p in singles + doubles for g in p.split("+")})
rng = np.random.default_rng(0)
tables = {
    "geneformer": {g: E[token_dict[name_to_id[g]]] for g in pgenes},
    "random":     {g: rng.normal(size=E.shape[1]) * E.std() for g in pgenes},
    "one_hot":    {g: np.eye(len(pgenes))[i] for i, g in enumerate(pgenes)},
}
feat = lambda p, tab: sum(tab[g] for g in p.split("+"))

train = singles + [d for d in doubles if d not in test_doubles]
test = list(test_doubles)
Y = lambda ps: np.vstack([pb.loc[p].values - ctrl for p in ps])
Ytr, Yte = Y(train), Y(test)

def score(pred, truth):
    rmse = np.sqrt(((pred - truth) ** 2).mean(1))
    dcor = np.array([corr(p, t) for p, t in zip(pred, truth)])
    return rmse, dcor

add_pred = np.vstack([(pb.loc[p.split("+")[0]].values - ctrl) + (pb.loc[p.split("+")[1]].values - ctrl) for p in test])
rm_add, dc_add = score(add_pred, Yte)

rows = [{"method": "additive (reference)", "rmse": rm_add.mean(), "delta_corr": np.nanmean(dc_add), "beats_additive_%": np.nan}]
for name, tab in tables.items():
    Xtr = np.vstack([feat(p, tab) for p in train])
    Xte = np.vstack([feat(p, tab) for p in test])
    ridge = RidgeCV(alphas=np.logspace(-3, 5, 17)).fit(Xtr, Ytr)
    rm, dc = score(ridge.predict(Xte), Yte)
    rows.append({"method": f"ridge + {name}", "rmse": rm.mean(), "delta_corr": np.nanmean(dc),
                 "beats_additive_%": 100 * float((rm < rm_add).mean())})

out = pd.DataFrame(rows).round(4)
print(out.to_string(index=False))
print("\nreference: estimated noise floor ~0.017 | no-change RMSE on test half 0.0622")
out.to_csv("/kaggle/working/cellcheck/geneformer_single_split_norman.csv", index=False)

## 7. Controls: scaled additive, shuffled embeddings, significance on one split

Scaled additive shrinks the additive prediction by one number learned on training doubles only.
Shuffled embeddings give every gene the Geneformer vector of a different gene, repeated 20 times.

In [ ]:
import numpy as np, pandas as pd
from scipy.stats import wilcoxon

def fit_eval(tab):
    Xtr = np.vstack([feat(p, tab) for p in train])
    Xte = np.vstack([feat(p, tab) for p in test])
    ridge = RidgeCV(alphas=np.logspace(-3, 5, 17)).fit(Xtr, Ytr)
    return score(ridge.predict(Xte), Yte)

# 1) scaled additive: one scalar learned on the TRAIN doubles only
train_doubles = [d for d in doubles if d not in test_doubles]
add_delta = lambda p: (pb.loc[p.split("+")[0]].values - ctrl) + (pb.loc[p.split("+")[1]].values - ctrl)
A = np.vstack([add_delta(d) for d in train_doubles])
B = np.vstack([pb.loc[d].values - ctrl for d in train_doubles])
s = float((A * B).sum() / (A ** 2).sum())
rm_scaled, dc_scaled = score(s * add_pred, Yte)
print(f"scaled additive: scale = {s:.3f} | RMSE {rm_scaled.mean():.4f} | delta_corr {np.nanmean(dc_scaled):.4f}")

# 2) real Geneformer vs shuffled-gene Geneformer vs random, 20 seeds each
rm_gf, _ = fit_eval(tables["geneformer"])
shuf, rand = [], []
for k in range(20):
    r = np.random.default_rng(100 + k)
    perm = r.permutation(len(pgenes))
    tab_s = {g: tables["geneformer"][pgenes[perm[i]]] for i, g in enumerate(pgenes)}
    tab_r = {g: r.normal(size=E.shape[1]) * E.std() for g in pgenes}
    shuf.append(fit_eval(tab_s)[0]); rand.append(fit_eval(tab_r)[0])
shuf, rand = np.array(shuf), np.array(rand)

def summ(name, arr):
    per_seed = arr.mean(1)
    print(f"{name:<24} RMSE {per_seed.mean():.4f} +/- {per_seed.std():.4f} (over 20 seeds)")
print(f"{'real Geneformer':<24} RMSE {rm_gf.mean():.4f}")
summ("shuffled Geneformer", shuf)
summ("random embeddings", rand)

# 3) paired test on the 62 doubles: real vs the average shuffled control
ctrl_rm = shuf.mean(0)
stat, p = wilcoxon(rm_gf, ctrl_rm)
print(f"\nreal beats shuffled on {100*float((rm_gf < ctrl_rm).mean()):.0f}% of doubles | Wilcoxon p = {p:.4f}")
stat2, p2 = wilcoxon(rm_gf, rm_scaled)
print(f"real vs scaled-additive: wins {100*float((rm_gf < rm_scaled).mean()):.0f}% | Wilcoxon p = {p2:.4f}")

pd.DataFrame({"double": test, "geneformer": rm_gf, "shuffled_mean": ctrl_rm,
              "scaled_additive": rm_scaled, "additive": rm_add}) \
  .to_csv("/kaggle/working/cellcheck/geneformer_controls_norman.csv", index=False)

## 8. Thirty random splits

The single-split result above was not stable enough to trust, so the whole comparison is repeated over 30 random splits.
The intervals are approximate because the splits overlap.

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import RidgeCV

def run_split(seed):
    r = np.random.default_rng(seed)
    te = sorted(r.choice(doubles, size=len(doubles) // 2, replace=False).tolist())
    trd = [d for d in doubles if d not in te]
    tr = singles + trd
    Ytr_, Yte_ = Y(tr), Y(te)
    rm = lambda P: float(np.sqrt(((P - Yte_) ** 2).mean(1)).mean())

    add_ = np.vstack([add_delta(d) for d in te])
    A_ = np.vstack([add_delta(d) for d in trd])
    B_ = np.vstack([pb.loc[d].values - ctrl for d in trd])
    s_ = float((A_ * B_).sum() / (A_ ** 2).sum())

    def fe(tab):
        Xtr_ = np.vstack([feat(p, tab) for p in tr])
        Xte_ = np.vstack([feat(p, tab) for p in te])
        rg = RidgeCV(alphas=np.logspace(-3, 5, 17)).fit(Xtr_, Ytr_)
        return rm(rg.predict(Xte_))

    gf = tables["geneformer"]
    shuf = []
    for _ in range(2):
        perm = r.permutation(len(pgenes))
        shuf.append(fe({g: gf[pgenes[perm[i]]] for i, g in enumerate(pgenes)}))
    rand_tab = {g: r.normal(size=E.shape[1]) * E.std() for g in pgenes}

    return {"seed": seed, "no_change": rm(np.zeros_like(Yte_)), "additive": rm(add_),
            "scaled_additive": rm(s_ * add_), "scale": s_,
            "geneformer": fe(gf), "shuffled": float(np.mean(shuf)), "random": fe(rand_tab)}

res_rows = []
for i, seed in enumerate(range(1000, 1030)):
    res_rows.append(run_split(seed))
    if (i + 1) % 10 == 0:
        print("splits done:", i + 1)
R = pd.DataFrame(res_rows)

FLOOR = 0.0171   # estimated noise floor from section 5
cols = ["no_change", "additive", "scaled_additive", "random", "shuffled", "geneformer"]
summary = pd.DataFrame({
    "mean_rmse": R[cols].mean(),
    "std_across_splits": R[cols].std(),
    "gap_to_floor_closed_%": 100 * (R["additive"].mean() - R[cols].mean()) / (R["additive"].mean() - FLOOR),
}).round(4)
print(summary)

d1 = R["shuffled"] - R["geneformer"]           # positive = Geneformer better
d2 = R["scaled_additive"] - R["geneformer"]
for name, d in [("Geneformer vs shuffled", d1), ("Geneformer vs scaled additive", d2)]:
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"\n{name}: mean improvement {d.mean():.4f} | 95% of splits between {lo:.4f} and {hi:.4f} "
          f"| Geneformer better in {100 * float((d > 0).mean()):.0f}% of splits")
print("scale learned across splits: mean", round(R['scale'].mean(), 3), "| min", round(R['scale'].min(), 3), "| max", round(R['scale'].max(), 3))

R.to_csv("/kaggle/working/cellcheck/geneformer_30splits_norman.csv", index=False)
summary.to_csv("/kaggle/working/cellcheck/geneformer_30splits_summary.csv")